# InfoNCE from scratch 


$$\mathcal{L}_N = -\mathbb{E}_X \left[ \log \frac{f_k(x_{t+k},\ c_t)}{\sum_{x_j \in X} f_k(x_j,\ c_t)} \right]$$

- $z_{t+k} = g_{\text{enc}}(x_{t+k}) \in \mathbb{R}^{d}$ : latent of the **future** observation 
- $c_t = g_{\text{ar}}(z_{\leq t}) \in \mathbb{R}^{d}$ : **context** vector produced by the autoregressive model
- $X = \{x_{t+k}\} \cup X_{\text{neg}}$ : 1 positive $x_{t+k} \sim p(x_{t+k} \mid c_t)$ + $N-1$ negatives drawn from $p(x_{t+k})$
- $f_k(x_{t+k}, c_t) = \exp(z_{t+k}^T W_k\, c_t)$ : **bilinear** score function 
- $W_k \in \mathbb{R}^{d \times d}$ : learned prediction matrix for step $k$

Learn representations $c_t$ that capture as much information as possible about future observations $x_{t+k}$.
The quantity to maximize is the **mutual information**:

$$I(x_{t+k};\ c_t) = \sum_{x,c} p(x_{t+k} \cap c_t) \log \frac{p(x_{t+k} \cap c_t)}{p(x_{t+k})\, p(c_t)}$$

$$= \sum_{x,c} p(x_{t+k} \cap c_t) \log \frac{p(x_{t+k} \mid c_t)}{p(x_{t+k})}$$
This is intractable to compute directly : we build a **lower bound** that can be maximized in practice.


## Reframe as a classification problem

Draw $N$ samples $X = \{x_1, \dots, x_N\}$:
- **1 positive** $x_i \sim p(x_{t+k} \mid c_t)$ : the true future observation
- **$N-1$ negatives** $x_{j \neq i} \sim p(x_{t+k})$ : distractors drawn from the marginal

Define $d \in \{1, \dots, N\}$ as the random variable indicating the index of the positive, with uniform prior $p(d=i) = \frac{1}{N}$.

## Computing the Bayes optimal classifier

We want $p(d=i \mid X \cap c_t)$.

**Bayes rule:**

$$p(d=i \mid X \cap c_t) = \frac{p(d=i \cap X \mid c_t)}{p(X \mid c_t)}$$

**Numerator — chain rule:**

$$p(d=i \cap X \mid c_t) = p(d=i \mid c_t) \cdot p(X \mid d=i \cap c_t)$$

$d$ is independent of $c_t$, so $p(d=i \mid c_t) = \frac{1}{N}$.

Given $d=i \cap c_t$, the $N$ samples are independent:

$$p(X \mid d=i \cap c_t) = p(x_i \mid c_t) \cdot \prod_{j \neq i} p(x_j)$$

Therefore:

$$p(d=i \cap X \mid c_t) = \frac{1}{N} \cdot p(x_i \mid c_t) \cdot \prod_{j \neq i} p(x_j)$$

**Denominator — law of total probability:**

$\{d=1\}, \dots, \{d=N\}$ form a partition, so:

$$p(X \mid c_t) = \sum_{m=1}^{N} p(d=m \cap X \mid c_t) = \frac{1}{N} \sum_{m=1}^{N} p(x_m \mid c_t) \cdot \prod_{j \neq m} p(x_j)$$

**Applying Bayes — the $\frac{1}{N}$ cancels:**

$$p(d=i \mid X \cap c_t) = \frac{p(x_i \mid c_t) \cdot \displaystyle\prod_{j \neq i} p(x_j)}{\displaystyle\sum_{m=1}^{N} p(x_m \mid c_t) \cdot \prod_{j \neq m} p(x_j)}$$

**Simplification — divide by $\prod_{j=1}^N p(x_j)$:**

$$\frac{p(x_i \mid c_t) \cdot \prod_{j \neq i} p(x_j)}{p(x_i) \cdot \prod_{j \neq i} p(x_j)} = \frac{p(x_i \mid c_t)}{p(x_i)}$$

Result:

$$\boxed{p(d=i \mid X \cap c_t) = \frac{\dfrac{p(x_i \mid c_t)}{p(x_i)}}{\displaystyle\sum_{m=1}^{N} \dfrac{p(x_m \mid c_t)}{p(x_m)}}}$$

---

## The InfoNCE loss 

Introduce a score function $f_k$ and define the predicted distribution:

$$q_\ell = \frac{f_k(x_\ell,\ c_t)}{\displaystyle\sum_{j=1}^{N} f_k(x_j,\ c_t)}$$

The target distribution is a one-hot on the positive: $p_\ell = \mathbf{1}[\ell=i]$.

The **cross-entropy** between $p$ and $q$:

$$H(p,\ q) = -\sum_{\ell=1}^{N} p_\ell \log q_\ell = -\log q_i$$

The InfoNCE loss is the expectation of this cross-entropy:

$$\mathcal{L}_N = \mathbb{E}_{X,\ c_t}\left[-\log \frac{f_k(x_i,\ c_t)}{\displaystyle\sum_{j=1}^{N} f_k(x_j,\ c_t)}\right]$$

---

## The optimal $f_k$ is the density ratio 

Using the identity $H(p, q) = H(p) + KL(p \| q)$.

**$H(p) = 0$** since $p$ is a one-hot — zero entropy. Therefore:

$$\mathcal{L}_N = \underbrace{\mathbb{E}[H(p)]}_{= 0} + \mathbb{E}[KL(p \| q)]$$

Minimizing $\mathcal{L}_N$ is equivalent to minimizing $\mathbb{E}[KL(p \| q)]$.

Since $KL \geq 0$ with equality iff $q = p$, the minimum is achieved when:

$$q_\ell = p(d=\ell \mid X \cap c_t) \quad \forall \ell$$

Substituting the Bayes-optimal classifier computed above:

$$\frac{f_k(x_\ell,\ c_t)}{\displaystyle\sum_j f_k(x_j,\ c_t)} = \frac{p(x_\ell \mid c_t)/p(x_\ell)}{\displaystyle\sum_j p(x_j \mid c_t)/p(x_j)}$$

This equality holds as soon as:

$$\boxed{f_k(x,\ c_t) \propto \frac{p(x \mid c_t)}{p(x)}}$$

The multiplicative constant cancels in the softmax. In practice, the paper parameterizes
$f_k(x_{t+k}, c_t) = \exp(z_{t+k}^T W_k c_t)$ and trains the system so that this bilinear
product approximates the log-density ratio.

---

## Lower bound on mutual information 

Substitute $f_k^* = p(x \mid c_t)/p(x)$ into the loss. Write $r_m = p(x_m \mid c_t)/p(x_m)$.

**Equation 6 — direct substitution:**

$$\mathcal{L}_N^{\text{opt}} = -\mathbb{E}_X \log \left[ \frac{r_i}{r_i + \displaystyle\sum_{x_j \in X_{\text{neg}}} r_j} \right]$$

**Equation 7 — algebra (factor out $r_i$ from denominator):**

$$-\log \frac{r_i}{r_i + \sum_j r_j} = \log\left(1 + \frac{1}{r_i}\sum_j r_j\right) = \log\left(1 + \frac{p(x_{t+k})}{p(x_{t+k} \mid c_t)} \sum_{x_j \in X_{\text{neg}}} \frac{p(x_j \mid c_t)}{p(x_j)}\right)$$

$$\mathcal{L}_N^{\text{opt}} = \mathbb{E}_X \log\left[1 + \frac{p(x_{t+k})}{p(x_{t+k} \mid c_t)} \sum_{x_j \in X_{\text{neg}}} \frac{p(x_j \mid c_t)}{p(x_j)}\right]$$

**Equation 8 — law of large numbers approximation:**

The $N-1$ negatives are i.i.d. from $p(x)$. By the LLN the empirical sum converges to $(N-1)$ times the expectation:

$$\mathcal{L}_N^{\text{opt}} \approx \mathbb{E}_X \log\left[1 + \frac{p(x_{t+k})}{p(x_{t+k} \mid c_t)}\ (N-1)\ \mathbb{E}_{x_j}\frac{p(x_j \mid c_t)}{p(x_j)}\right]$$

**Equation 9 — compute the expectation:**

$$\mathbb{E}_{x_j \sim p(x)}\left[\frac{p(x_j \mid c_t)}{p(x_j)}\right] = \sum_x p(x) \cdot \frac{p(x \mid c_t)}{p(x)} = \sum_x p(x \mid c_t) = 1$$

$$\mathcal{L}_N^{\text{opt}} \approx \mathbb{E}_X \log\left[1 + \frac{p(x_{t+k})}{p(x_{t+k} \mid c_t)}\ (N-1)\right]$$

**Equation 10 — inequality via AM-GM:**

Write $b = \frac{p(x_{t+k})}{p(x_{t+k}|c_t)} = \frac{1}{r_i}$. AM-GM on $(1,\ \underbrace{b, \dots, b}_{N-1})$:

$$\frac{1 + (N-1)b}{N} \geq b^{\frac{N-1}{N}} \implies \log(1 + (N-1)b) \geq \log N + \frac{N-1}{N}\log b$$

Taking the expectation over $x_{t+k} \sim p(x \mid c_t)$ and using
$\mathbb{E}[\log b] = -KL(p(x|c)\|p(x)) \leq 0$:

$$\mathbb{E}[\log(1+(N-1)b)] \geq \log N - \frac{N-1}{N}KL \geq \log N - KL = \mathbb{E}\left[\log\frac{N}{r_i}\right]$$

$$\mathcal{L}_N^{\text{opt}} \geq \mathbb{E}_X \log\left[\frac{p(x_{t+k})}{p(x_{t+k} \mid c_t)} \cdot N\right]$$

**Equation 11 — final computation:**

$$\mathbb{E}_{x_{t+k} \sim p(x|c_t)}\left[\log \frac{p(x_{t+k})}{p(x_{t+k} \mid c_t)} \cdot N\right] = \log N + \mathbb{E}\left[\log \frac{p(x_{t+k})}{p(x_{t+k} \mid c_t)}\right] = \log N - I(x_{t+k};\ c_t)$$

Therefore:

$$\mathcal{L}_N^{\text{opt}} \geq \log N - I(x_{t+k};\ c_t)$$

$$\boxed{I(x_{t+k};\ c_t) \geq \log N - \mathcal{L}_N}$$

**Interpretation:** minimizing $\mathcal{L}_N$ maximizes a lower bound on $I$. The larger $N$, the tighter the bound.
As $N \to \infty$, the LLN approximation becomes exact and the bound is tight.